In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import glob

# Create the output directory if it doesn't exist
folder = 'sweep_results_linear/MSE/again/'
output_dir = f'figures/{folder}'
os.makedirs(output_dir, exist_ok=True)

# Get the list of CSV files in the results/sweep_results directory
csv_files = glob.glob(f'results/{folder}/sweep_params_*.csv')

# Iterate over each CSV file
for file_path in csv_files:
    # Extract the variable name from the file name
    var = '_'.join(os.path.basename(file_path).split('_')[2:]).split('.')[0]

    # Load the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
    # filter out the rows where the run was not successful with accuracy < 0.8
    print(f'Unconverged runs {var}: {len(df[df["res_accuracy"] < 0.95])}/{len(df)}\n')
    # df = df[df['res_accuracy'] > 0.95]

    res_cols = ['loss'] + [col_name for col_name in list(df.columns.values) if col_name.split('_')[0] == 'res']

    # Create a figure with 3 subplots
    fig, axs = plt.subplots(len(res_cols)//3+1, 3, figsize=(30, 5*len(res_cols)/3))
    fig.suptitle(f'Unconverged: {len(df[df["res_accuracy"] < 0.95])}/{len(df)}\n')
    # Plot boxplots for each value variable
    for ax, col_name in zip(axs.flatten(), res_cols):
        res_txt = '_'.join(col_name.split('_')[1:])
        df[col_name] = abs(df[col_name])
        # df[var] = np.round(df[var], 2) if df[var].dtype == 'float64' else df[var]
        sns.boxplot(data=df, x=var, y=col_name, ax=ax)
        ax.set_title(f'{var} vs {res_txt}')
        ax.set_xlabel(var)
        ax.set_ylabel(res_txt)
        if 'NC1' in col_name:
            ax.set_yscale('log')
        elif df[col_name].max() <= 1:
            ax.set_ylim(0, 1)
        else:
            ax.set_ylim(0, df[col_name].max()*1.1)

    # Adjust layout and save the figure
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir, f'{var}.png'))
    plt.close(fig)

Unconverged runs one_hot_actions: 0/20

Unconverged runs sig_h_2: 0/100

Unconverged runs allow_backwards: 0/12

Unconverged runs fixed_output: 0/20



1. Why after L=13 there is no order? No other variables shows this behavior. Weird non-monotonic behavior in that sense.
2. Need a better metric for corridors centered with each other. Maybe project dx to their plane
3. Sweet spot at G=.85?
4. Signal peak at L=3

1. Linear network
2. Check sweep when allow backwards = F
3. Single corridor only. When test on second corridor - still in same dimension?
4. n corridors
5. D-dimensional corridors
6. Split action True and vary G.